# **Softmax Regression**

## **Libraries**

In [115]:
import tensorflow as tf
import numpy as np

class DataSets:
    def __init__(self, train_images, train_labels, test_images, test_labels):
        self.train = self.Data(train_images, train_labels)
        self.test = self.Data(test_images, test_labels)

    class Data:
        def __init__(self, images, labels):
            self.images = images
            self.labels = labels
            self.num_examples = images.shape[0]

        def next_batch(self, batch_size):
            idx = np.random.choice(self.num_examples, batch_size, replace=False)
            return self.images[idx], self.labels[idx]

def read_data_sets(path="./", one_hot=True):
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

    # Reshape & normalize
    x_train = x_train.reshape(-1, 784).astype(np.float32) / 255.0
    x_test = x_test.reshape(-1, 784).astype(np.float32) / 255.0

    if one_hot:
        y_train = tf.keras.utils.to_categorical(y_train, 10)
        y_test = tf.keras.utils.to_categorical(y_test, 10)

    return DataSets(x_train, y_train, x_test, y_test)


In [116]:
import tensorflow as tf

## **Dataset**

In [117]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize the images to [0,1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

# Flatten if using a fully-connected network
x_train_flat = x_train.reshape(-1, 28*28)
x_test_flat  = x_test.reshape(-1, 28*28)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

x_train shape: (60000, 28, 28)
y_train shape: (60000,)


## **Settings**

In [118]:
# Hyperparameters
learning_rate = 0.5
training_epochs = 30
batch_size = 256

n_features = x_train.shape[1] * x_train.shape[2]  # 28*28 = 784
n_classes = len(np.unique(y_train))

## **Graph definition**

In [119]:
g = tf.Graph()
with g.as_default():

    tf_x = tf.compat.v1.placeholder(tf.float32, [None, n_features])
    tf_y = tf.compat.v1.placeholder(tf.float32, [None, n_classes])

    params = {
        'weights': tf.Variable(tf.zeros(shape=[n_features, n_classes],
                                               dtype=tf.float32), name='weights'),
        'bias': tf.Variable([[n_classes]], dtype=tf.float32, name='bias')}

    linear = tf.matmul(tf_x, params['weights']) + params['bias']
    pred_proba = tf.nn.softmax(linear, name='predict_probas')
    
    cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(
        logits=linear, labels=tf_y), name='cost')
    optimizer = tf.compat.v1.train.GradientDescentOptimizer(learning_rate=learning_rate)
    train = optimizer.minimize(cost, name='train')

    pred_labels = tf.argmax(pred_proba, 1, name='predict_labels')
    correct_prediction = tf.equal(tf.argmax(tf_y, 1), pred_labels)
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32), name='accuracy')

## **Training and evaluation**

In [120]:
def next_batch(X, y, batch_size):
    idx = np.random.choice(X.shape[0], batch_size, replace=False)
    return X[idx], y[idx]

def one_hot(labels, num_classes):
    return np.eye(num_classes)[labels]

num_classes = 10
y_train_oh = np.eye(num_classes)[y_train]
y_test_oh  = np.eye(num_classes)[y_test]

In [122]:
tf.compat.v1.disable_eager_execution()

with tf.compat.v1.Session(graph=g) as sess:
    sess.run(tf.compat.v1.global_variables_initializer())

    for epoch in range(training_epochs):
        avg_cost = 0.
        total_batch = x_train.shape[0] // batch_size

        for i in range(total_batch):
            batch_x, batch_y_int = next_batch(x_train_flat, y_train, batch_size)
            batch_y = one_hot(batch_y_int, num_classes=10)
            _, c = sess.run(['train', 'cost:0'], feed_dict={tf_x: batch_x, tf_y: batch_y})
            avg_cost += c
        
        train_acc = sess.run('accuracy:0', feed_dict={tf_x: x_train_flat,
                                              tf_y: y_train_oh})
        valid_acc = sess.run('accuracy:0', feed_dict={tf_x: x_test_flat,
                                              tf_y: y_test_oh})
        
        print("Epoch: %03d | AvgCost: %.3f" % (epoch + 1, avg_cost / (i + 1)), end="")
        print(" | Train/Valid ACC: %.3f/%.3f" % (train_acc, valid_acc))
        
    test_acc = sess.run(accuracy, feed_dict={tf_x: x_test_flat, tf_y: y_test_oh})
    print('Test ACC: %.3f' % test_acc)

I0000 00:00:1759152527.158586      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch: 001 | AvgCost: 0.466 | Train/Valid ACC: 0.903/0.907
Epoch: 002 | AvgCost: 0.329 | Train/Valid ACC: 0.912/0.916
Epoch: 003 | AvgCost: 0.310 | Train/Valid ACC: 0.914/0.915
Epoch: 004 | AvgCost: 0.299 | Train/Valid ACC: 0.911/0.910
Epoch: 005 | AvgCost: 0.294 | Train/Valid ACC: 0.918/0.918
Epoch: 006 | AvgCost: 0.293 | Train/Valid ACC: 0.919/0.919
Epoch: 007 | AvgCost: 0.285 | Train/Valid ACC: 0.922/0.920
Epoch: 008 | AvgCost: 0.281 | Train/Valid ACC: 0.922/0.923
Epoch: 009 | AvgCost: 0.280 | Train/Valid ACC: 0.923/0.923
Epoch: 010 | AvgCost: 0.274 | Train/Valid ACC: 0.923/0.921
Epoch: 011 | AvgCost: 0.280 | Train/Valid ACC: 0.924/0.921
Epoch: 012 | AvgCost: 0.275 | Train/Valid ACC: 0.925/0.921
Epoch: 013 | AvgCost: 0.276 | Train/Valid ACC: 0.924/0.922
Epoch: 014 | AvgCost: 0.275 | Train/Valid ACC: 0.926/0.921
Epoch: 015 | AvgCost: 0.273 | Train/Valid ACC: 0.926/0.922
Epoch: 016 | AvgCost: 0.271 | Train/Valid ACC: 0.927/0.921
Epoch: 017 | AvgCost: 0.268 | Train/Valid ACC: 0.927/0.9